In [ ]:
Oracle AI Data Platform v1.0

Copyright © 2025, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Jira Cloud Connector Samples

Read-only ingestion samples for Jira Cloud using the `aidp-connector-jira` package and the Jira REST API. Replace all placeholders before running a sample.

## Prerequisites

1. **Connector package.** In a clone of [arbisoft/oracle-aidp-connectors](https://github.com/arbisoft/oracle-aidp-connectors), in `src/aidp_connector_jira`, run:
   ```
   uv build
   ```
   It builds `dist/aidp_connector_jira-<version>-py3-none-any.whl`. Upload it to a workspace folder, install it from the cluster's **Library** tab → **Install Library** → **Workspace**, then restart the cluster.
2. **Credential.** Create a **Secret Token** credential in the Credential Store with keys `site`, `email` and `token`; the token comes from https://id.atlassian.com/manage/api-tokens.
3. **Network.** The cluster needs HTTPS access to your Jira Cloud site.

## Configuration

The issues to read and the Delta table to load them into.

In [ ]:
from aidp_connector_jira import format_summary, parse_config, preview, read_credentials, run

config = parse_config({
    "jira": {"credential_name": "<CREDENTIAL_NAME>"},  # Secret Token credential with keys site, email, token
    "target": {"catalog": "<CATALOG>", "schema": "<SCHEMA>", "table": "<TABLE_NAME>"},
    "sync": {
        "jql": "<JQL_QUERY>",  # for example: project = <PROJECT_KEY>; no ORDER BY
        "extra_fields": [],    # other Jira fields, e.g. custom fields, stored as JSON in raw_fields
    },
})

## Credentials

Reads the site, email and API token from the Credential Store; the site must be a Jira Cloud host.

In [ ]:
creds = read_credentials(config.jira, aidputils.secrets.get)
print("credentials read (not shown)")

## Ingestion Sample

Previews the first 10 issues that match the query; only the pages needed for them are fetched.

In [ ]:
jira_df = preview(spark, config, creds)

jira_df.show()

## Incremental Load into Delta

Loads every matching issue on the first run, then only issues updated since the last run, merged on `key`. A full refresh reloads every matching issue and overwrites the table, which also removes issues deleted in Jira.

In [ ]:
summary = run(spark, config, creds)  # run(spark, config, creds, "full") for a full refresh

print(format_summary(summary))

## Connector Options

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| `jira.credential_name` | Credential Store entry | Y | Entry with keys `site`, `email` and `token`. |
| `target.catalog` | Existing catalog | Y | Target catalog. |
| `target.schema` | Schema name | Y | Target schema. Created if missing. |
| `target.table` | Table name | Y | Target table. |
| `sync.jql` | JQL without ORDER BY | Y | Which issues to load. |
| `sync.extra_fields` | List of Jira field IDs | N | Other fields to load, as JSON in `raw_fields`. |
| `sync.page_size` | 1 to 5000 | N | Issues per search request. Default `100`. |
| `sync.overlap_seconds` | Integer, 0 or more | N | How far before the watermark a later run re-reads. Default `300`. |